In [1]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from model import ImprovedCNN

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"device:{device}")

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),
                         (0.5,0.5,0.5))
])

train_dataset = datasets.CIFAR10(
    "./data",
    train=True,
    download=True,
    transform=train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)

model = ImprovedCNN().to(device)
print(next(model.parameters()).device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.1
)

epochs = 30

loss_history = []

for epoch in range(epochs):

    model.train()
    running_loss = 0

    # start = time.time()

    for images, labels in train_loader:

        # print("Load batch:", time.time() - start)

        # with pin_memory=True
        # images = images.to(device, non_blocking=True)
        # labels = labels.to(device, non_blocking=True)
        
        # without pin_memory=True
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    epoch_loss = running_loss / len(train_loader)
    loss_history.append(epoch_loss)

    print(
        f"Epoch {epoch+1}/{epochs} Loss: {epoch_loss:.4f}"
    )


In [ ]:
torch.save(
    model.state_dict(),
    "cifar10_improved_cnn.pth"
)


plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.savefig(
    "results/loss_curve.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()


In [ ]:
print(sum(p.numel() for p in model.parameters())/1e6, "M parameters")